In [3]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression, GammaRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder

from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import r2_score

import pickle
import json
import matplotlib.pyplot as plt
%matplotlib inline

# Загрузка модели

In [4]:
def load_model(model_path='model.pkl'):
    with open(model_path, 'rb') as f:
        model = pickle.load(f)
    return model

loaded_model = load_model()
print(f"Модель загружена: {loaded_model}")

Модель загружена: KNeighborsRegressor(n_neighbors=4, p=1)


# Преобразование данных

In [5]:
with open('config.json', 'r', encoding='utf-8') as f:
    config = json.load(f)

In [10]:
def prepare_data(data, config):
    df = pd.read_csv(data, index_col='car_ID')
    X = df.drop(['price'], axis=1)

    car_names = X['CarName'].str.split(' ', expand=True).fillna('')
    X['car_company'] = car_names[0]
    X['car_model'] = np.sum(car_names.loc[:, 1:], axis=1)

    X.drop(['CarName'], axis=1, inplace=True)
    
    num_cols = config['num_cols']
    str_cols = config['str_cols']
    bin_cols = config['bin_cols']
    cat_cols = config['cat_cols']

    X['car_company'] = X['car_company'].replace(
        {
            'maxda': 'mazda',
            'porcshce': 'porsche',
            'Nissan': 'nissan',
            'vokswagen': 'volkswagen',
            'vw': 'volkswagen',
            'toyouta': 'toyota',
        }
    )

    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

    X_train_encoded = enc.fit_transform(X[bin_cols + cat_cols])
    X[bin_cols + cat_cols] = X_train_encoded

    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    return X

In [12]:
data_to_predict = prepare_data('data/carprice_gen.csv',config=config)

In [15]:
predict = loaded_model.predict(data_to_predict)

C:\Users\HokoriDynasty\AppData\Roaming\Python\Python311\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] Не удается найти указанный файл
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\HokoriDynasty\AppData\Roaming\Python\Python311\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Program Files\Python311\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Program Files\Python311\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Program Files\Python311\Lib\subprocess.py", line 1538, in _exec

In [16]:
predict

array([ 7082.  , 10583.75, 10487.5 , 15363.5 ,  7351.5 ,  9278.25,
       16898.75, 13939.5 , 14626.  , 12567.5 ])